# A short introduction to containerized software

After spending time using nf-core pipelines to answer bioinformatic questions, we will focus on the processes that lie behind these pipelines now.

Today, we will focus on containerization, namely via Docker. 



1. Check if Docker is installed.

In [2]:
!docker info

Client:
 Version:    29.8.0
 Context:    default
 Debug Mode: false
 Plugins:
  agent: Docker AI Agent Runner (Docker Inc.)
    Version:  v1.140.0
    Path:     /usr/local/lib/docker/cli-plugins/docker-agent
  ai: Docker AI Agent - Ask Gordon (Docker Inc.)
    Version:  v1.31.0
    Path:     /usr/local/lib/docker/cli-plugins/docker-ai
  buildx: Docker Buildx (Docker Inc.)
    Version:  v0.37.1
    Path:     /usr/local/lib/docker/cli-plugins/docker-buildx
  compose: Docker Compose (Docker Inc.)
    Version:  v5.5.1
    Path:     /usr/local/lib/docker/cli-plugins/docker-compose
  debug: Get a shell into any image or container (Docker Inc.)
    Version:  0.0.47
    Path:     /usr/local/lib/docker/cli-plugins/docker-debug
  desktop: Docker Desktop commands (Docker Inc.)
    Version:  v0.4.4
    Path:     /usr/local/lib/docker/cli-plugins/docker-desktop
  dhi: CLI for managing Docker Hardened Images (Docker Inc.)
    Version:  v0.0.7
    Path:     /usr/local/lib/docker/cli-plugins/docker-dh

### What is a container?

A container is an isolated process for each of the app's component.
It is a lightweight, standalone, and executable unit of software that packages code and all its dependencies

### Why do we use containers?

We use containaers to ensure that the application runs consistently regardless of where it is deployed.

also we use containers to ensure reproducibility (ESTO SE TIENE QUE MEJORAR)

### What is a docker image?

A Docker image is a blueprint or template for creating containers. WE use them to define the environment, including the application code, libraries, dependencies, and configuration needed to run an application

### Let's run our first docker image:

### Login to docker

In [ ]:
# This you need to do on the command line directly
# docker login -u paulaparisbayarri (my user name)

### Run your first docker container

In [15]:
!docker run hello-world


Hello from Docker!
This message shows that your installation appears to be working correctly.

To generate this message, Docker took the following steps:
 1. The Docker client contacted the Docker daemon.
 2. The Docker daemon pulled the "hello-world" image from the Docker Hub.
    (amd64)
 3. The Docker daemon created a new container from that image which runs the
    executable that produces the output you are currently reading.
 4. The Docker daemon streamed that output to the Docker client, which sent it
    to your terminal.

To try something more ambitious, you can run an Ubuntu container with:
 $ docker run -it ubuntu bash

Share images, automate workflows, and more with a free Docker ID:
 https://hub.docker.com/

For more examples and ideas, visit:
 https://docs.docker.com/get-started/



### Find the container ID

In [16]:
!docker ps --all

CONTAINER ID   IMAGE         COMMAND    CREATED         STATUS                     PORTS     NAMES
076a5ad3ae27   hello-world   "/hello"   6 seconds ago   Exited (0) 5 seconds ago             sweet_sutherland
eb5b71f07c00   hello-world   "/hello"   3 days ago      Exited (0) 3 days ago                suspicious_wing
ec0939cc896d   hello-world   "/hello"   3 days ago      Exited (0) 3 days ago                silly_faraday


container ID: 076a5ad3ae27   

### Delete the container again, give prove its deleted

In [17]:
!docker rm 076a5ad3ae27      

print("Container has been sucessfully deleted!")

076a5ad3ae27
Container has been sucessfully deleted!


In [22]:
!docker ps --all

print("\nSee there's not the continer I previously created")

CONTAINER ID   IMAGE         COMMAND    CREATED      STATUS                  PORTS     NAMES
eb5b71f07c00   hello-world   "/hello"   3 days ago   Exited (0) 3 days ago             suspicious_wing
ec0939cc896d   hello-world   "/hello"   3 days ago   Exited (0) 3 days ago             silly_faraday

See there's not the continer I previously created


### FASTQC is a very useful tool as you've learned last week. Let's try and run it from command line

Link to the software: https://www.bioinformatics.babraham.ac.uk/projects/fastqc/

Please describe the steps you took to download and run the software for the example fastq file from last week below:

1. We collected the sample information from the paper, including the experimental condition and the number of reads for each sample
2. We combiend both informations (conditions and reads per sample) into a new DataFrame
3. We selected the two samples with the smallest number of reads, to reduce the computational load
4. We used nf-core/fetchngs with Nextflow to download the sequencing data for those two samples 
5. As a result, we obtained the corresponding paired-end FASTQ files for those samples.

### Very well, now let's try to make use of its docker container

1. create a container holding fastqc using seqera containers (https://seqera.io/containers/)
2. use the container to generate a fastqc html of the example fastq file

In [23]:
# pull the container
!docker pull community.wave.seqera.io/library/fastqc:0.12.1--9971ea336a9eddae

0.12.1--9971ea336a9eddae: Pulling from library/fastqc

0ce1546f: Pulling fs layer 
Digest: sha256:f97f7658a191bebe2f2a007cab5b815d380f89a948d6eb1481b046ab7618bebc[2A
Status: Downloaded newer image for community.wave.seqera.io/library/fastqc:0.12.1--9971ea336a9eddae
community.wave.seqera.io/library/fastqc:0.12.1--9971ea336a9eddae


In [24]:
!docker images

                                                            i Info →   U  In Use
IMAGE                           ID             DISK USAGE   CONTENT SIZE   EXTRA
community.wave.seqera.io/library/bioconductor-deseq2_bioconductor-limma:b56a0c9ddc3e87e1
                                e220c341c297       2.56GB          649MB        
community.wave.seqera.io/library/csvtk:0.37.0--113625988dd3285d
                                b13e22e5ecee        206MB           52MB        
community.wave.seqera.io/library/differentialabundance-report:10d7bd03a13e6ca9
                                796a346f0f7e       5.28GB         1.31GB        
community.wave.seqera.io/library/fastqc:0.12.1--9971ea336a9eddae
                                f97f7658a191       1.27GB          344MB        
community.wave.seqera.io/library/p7zip:16.02--94efedc6cfea7db9
                                b6307567044c        207MB         51.5MB        
community.wave.seqera.io/library/pandas_python:67bda66f0cb8a241
         

In [25]:
# run the container and save the results to a new "fastqc_results" directory
!docker run --rm \
    -v /home/paula/.ssh/computational-workflows-2026/day_02/SRA_runs/fastq:/data \
    -v /home/paula/.ssh/computational-workflows-2026/day_04/fastqc_results:/results \
    community.wave.seqera.io/library/fastqc:0.12.1--9971ea336a9eddae \
    fastqc /data/SRR23195511_1.fastq.gz -o /results

application/gzip
Started analysis of SRR23195511_1.fastq.gz
Approx 5% complete for SRR23195511_1.fastq.gz
Approx 10% complete for SRR23195511_1.fastq.gz
Approx 15% complete for SRR23195511_1.fastq.gz
Approx 20% complete for SRR23195511_1.fastq.gz
Approx 25% complete for SRR23195511_1.fastq.gz
Approx 30% complete for SRR23195511_1.fastq.gz
Approx 35% complete for SRR23195511_1.fastq.gz
Approx 40% complete for SRR23195511_1.fastq.gz
Approx 45% complete for SRR23195511_1.fastq.gz
Approx 50% complete for SRR23195511_1.fastq.gz
Approx 55% complete for SRR23195511_1.fastq.gz
Approx 60% complete for SRR23195511_1.fastq.gz
Approx 65% complete for SRR23195511_1.fastq.gz
Approx 70% complete for SRR23195511_1.fastq.gz
Approx 75% complete for SRR23195511_1.fastq.gz
Approx 80% complete for SRR23195511_1.fastq.gz
Approx 85% complete for SRR23195511_1.fastq.gz
Approx 90% complete for SRR23195511_1.fastq.gz
Approx 95% complete for SRR23195511_1.fastq.gz
Analysis complete for SRR23195511_1.fastq.gz


### Now that you know how to use a docker container, which approach was easier and which approach will be easier in the future?

I found using Docker container easier because I did not need to install FastQC and all its dependencies manually. I only had to pull the container and run it with the FASTQfile. In the future, I think Docker will also be easier, especially when using different bioinformatics tools, because container already include the required software and versions, which helps avoid installation and dependency problems.

### What would you say, which approach is more reproducible?

Using Docker containers is more reproducible because the container includes a specific version of the software and its dependencies. This makes it easier to run the same analysis in the same environment on different computers. 

### Compare the file to last weeks fastqc results, are they identical?
### Is the fastqc version identical?

The FastQC reports are very similar, but they do not seem to be completely identical. Analysis performed by nf-core/rnaseq was a multiqc, where all the FastQC of the different measurments in both reading frames(?) (forward and rverse reads) were analysed. However, when using docker, just a file from one of the measurments was performed. By analysing the quality performance, it seems that the one from docker performed better, since it has a greater score than the one performed with multiqc.

per base n content is different at the biginning: while multiqc has a peack at the first few base pairs, in the one perfromed with docker does not.

sequence duplication levels is also different in both reports: while in multiqc it starts lower(48%), and then it seems to have a peak that grows from >5k to >10k. however in docker's report it starts higher (around 55%) and the tail of the graph (the end part) does not raise. it remains more or less constant.

adapter content: the only one that changes significantly is illumina_universal_adaptar, which raises higher than the 50% in multiqc and around 25% in fastqc performed by docker.

## Dockerfiles

We now used Docker containers and images directly to boost our research. 

Let's create our own toy Dockerfile including the "cowsay" tool (https://en.wikipedia.org/wiki/Cowsay)

Hints:
1. Docker is Linux, so you need to know the apt-get command to install "cowsay"

In [27]:
# open the file "my_dockerfile" in a text editor

!docker build -f my_dockerfile -t cowsay_image .



[+] Building 0.0s (0/1)                                          docker:default
[+] Building 0.2s (1/2)                                          docker:default
 => [internal] load build definition from my_dockerfile                    0.0s
 => => transferring dockerfile: 717B                                       0.0s
 => [internal] load metadata for docker.io/library/debian:bullseye-slim    0.1s
[+] Building 0.3s (1/2)                                          docker:default
 => [internal] load build definition from my_dockerfile                    0.0s
 => => transferring dockerfile: 717B                                       0.0s
 => [internal] load metadata for docker.io/library/debian:bullseye-slim    0.3s
[+] Building 0.5s (1/2)                                          docker:default
 => [internal] load build definition from my_dockerfile                    0.0s
 => => transferring dockerfile: 717B                                       0.0s
 => [internal] load metadata for docke

### Explain the RUN and ENV lines you added to the file

The RUN instructions executes commands during the image build process. In this case, it is used to update the package list and install cowsay.

The ENV instruction sets an environment variable inside the image. In this case, it adds the directory containining cowsay to the PATH, so the command can be called directly without specifying its full path

In [28]:
# build the docker image
!docker build -f my_dockerfile -t cowsay-image .


[+] Building 0.0s (0/1)                                          docker:default
[+] Building 0.2s (1/3)                                          docker:default
 => [internal] load build definition from my_dockerfile                    0.0s
 => => transferring dockerfile: 717B                                       0.0s
 => [internal] load metadata for docker.io/library/debian:bullseye-slim    0.1s
 => [auth] library/debian:pull token for registry-1.docker.io              0.0s
[+] Building 0.3s (2/3)                                          docker:default
 => [internal] load build definition from my_dockerfile                    0.0s
 => => transferring dockerfile: 717B                                       0.0s
 => [internal] load metadata for docker.io/library/debian:bullseye-slim    0.3s
 => [auth] library/debian:pull token for registry-1.docker.io              0.0s
[+] Building 0.5s (2/3)                                          docker:default
 => [internal] load build definition fr

In [29]:
# make sure that the image has been built
!docker images

                                                            i Info →   U  In Use
IMAGE                           ID             DISK USAGE   CONTENT SIZE   EXTRA
community.wave.seqera.io/library/bioconductor-deseq2_bioconductor-limma:b56a0c9ddc3e87e1
                                e220c341c297       2.56GB          649MB        
community.wave.seqera.io/library/csvtk:0.37.0--113625988dd3285d
                                b13e22e5ecee        206MB           52MB        
community.wave.seqera.io/library/differentialabundance-report:10d7bd03a13e6ca9
                                796a346f0f7e       5.28GB         1.31GB        
community.wave.seqera.io/library/fastqc:0.12.1--9971ea336a9eddae
                                f97f7658a191       1.27GB          344MB        
community.wave.seqera.io/library/p7zip:16.02--94efedc6cfea7db9
                                b6307567044c        207MB         51.5MB        
community.wave.seqera.io/library/pandas_python:67bda66f0cb8a241
         

In [ ]:
# run the docker file 
!docker run --rm cowsay-image

## Let's do some bioinformatics with the docker file and create a new docker file that holds the salmon tool used in rnaseq

To do so, use "curl" in your new dockerfile to get salmon from https://github.com/COMBINE-lab/salmon/releases/download/v1.5.2/salmon-1.5.2_linux_x86_64.tar.gz

In [ ]:
# use the file "salmon_docker" in this directory to build a new docker image

In [1]:
# build the image
!docker build -f salmon_docker -t salmon_image .



[+] Building 0.0s (0/1)                                          docker:default
[+] Building 0.2s (1/2)                                          docker:default
 => [internal] load build definition from salmon_docker                    0.0s
 => => transferring dockerfile: 622B                                       0.0s
 => [internal] load metadata for docker.io/library/debian:bookworm-slim    0.2s
[+] Building 0.4s (1/2)                                          docker:default
 => [internal] load build definition from salmon_docker                    0.0s
 => => transferring dockerfile: 622B                                       0.0s
 => [internal] load metadata for docker.io/library/debian:bookworm-slim    0.3s
[+] Building 0.5s (1/2)                                          docker:default
 => [internal] load build definition from salmon_docker                    0.0s
 => => transferring dockerfile: 622B                                       0.0s
 => [internal] load metadata for docke

In [3]:
# run the docker image to give out the version of salmon
!docker run --rm salmon_image salmon --version

salmon 1.5.2


## Do you think bioinformaticians have to create a docker image every time they want to run a tool?

Find the salmon docker image online and run it on your computer.

What is https://biocontainers.pro/ ?

No, bioinformaticians do not need to create a Docker image every time they want to use a tool. Many bioinformatics tools already have pre-built containers available through repositories such as BioContainers.

BioContainers is a community-driven project that provides containerized scientific software to make tools easier to install and analyses more reproducible.

In [4]:
!docker pull quay.io/biocontainers/salmon:1.5.2--h84f40af_0

1.5.2--h84f40af_0: Pulling from biocontainers/salmon

59c44bf5: Pulling fs layer 
45ee6d5d: Pulling fs layer 
Digest: sha256:4ae09a47788f08317bd2f758ac4c8804c9e87d88caf500c449e18ac4794d0332[3A
Status: Downloaded newer image for quay.io/biocontainers/salmon:1.5.2--h84f40af_0
quay.io/biocontainers/salmon:1.5.2--h84f40af_0


In [5]:
!docker run --rm quay.io/biocontainers/salmon:1.5.2--h84f40af_0 salmon --version

salmon 1.5.2


## Are there other ways to create Docker (or Apptainer) images?

What is https://seqera.io/containers/ ?

Yes, there are multiple ways to create Docker or Apptainer images. For example, there are services, such as Seqera Containers that can generate container images automatically from Conda or PyPI packages. The user selecs the required software and container format, and Seqera builds the image and provides a URI that can be used directly.

Seqera Containers is a free service that generates ready-to-use container images from Conda or PyPI packages. 